# Benchmark setup and readiness

This notebook shows the prepared task coverage and setup evidence. It makes **no paid API calls**. Local scripted simulations validate execution; they are not research findings. Read `docs/BENCHMARKS.md` for the native/compatibility distinctions and paid-run commands.

In [1]:
from pathlib import Path
import sys, json, subprocess
import pandas as pd
from IPython.display import display
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'configs/benchmarks.json').exists())
assert Path(sys.prefix).resolve() == (ROOT / '.venv').resolve(), 'Select the project .venv kernel.'
sys.path.insert(0, str(ROOT))
print('Project:', ROOT)
print('Interpreter:', sys.executable)

Project: A:\paper\llm-business-shocks-sim
Interpreter: A:\paper\llm-business-shocks-sim\.venv\Scripts\python.exe


## Prepared coverage

Four Retail configurations each run 180 days. FirmBench uses its 507 included sampled reasoning cases, 10 Consulting cases, both 25-week Beer scenarios, and one 96-step EDT scenario. ICBE uses all six ordered model pairs for 100 rounds each. The chosen CoT/ReAct methods and substituted gateway models are documented; this is not an exact reproduction of published model scores.

In [2]:
config = json.loads((ROOT / 'configs/benchmarks.json').read_text())
from src.benchmarks.common import TASKS
display(pd.DataFrame({'task': TASKS}))
print('Primary model:', config['primary_model'])
print('Interviewer/judge:', config['judge_model'])
print('ICBE models:', ', '.join(config['icbe_models']))

,task
0,retail:dynamic_middle
1,retail:dynamic_hard
2,retail:still_middle
3,retail:still_hard
4,firm:FinCode
5,firm:CodeFinQA
6,firm:CodeTAT-QA
7,firm:SEC-NUM
8,firm:TAT-QA
9,firm:ConvFinQA


Primary model: gpt-4.1-mini-2025-04-14
Interviewer/judge: gpt-4o-mini-2024-07-18
ICBE models: gpt-4.1-mini-2025-04-14, gpt-4o-mini-2024-07-18, deepseek-v3.2


## Saved verification evidence

The snapshot records 20 local checks, including native data processing/scoring, simulation horizons and file export. The API check separately records actual response compatibility and token usage.

In [3]:
readiness = json.loads((ROOT / 'docs/results/benchmark_readiness.json').read_text())
display(pd.DataFrame([{'task': c['task'], 'status': c['status']} for c in readiness['checks']]))
assert all(c['status'] == 'passed' for c in readiness['checks'])
print('Verified at UTC:', readiness['verified_at_utc'])
print('Gateway compatibility usage:', readiness['gateway']['usage'])
display(pd.Series(readiness['dependency_versions'], name='installed version').to_frame())

,task,status
0,dependencies,passed
1,retail:dynamic_middle,passed
2,retail:dynamic_hard,passed
3,retail:still_middle,passed
4,retail:still_hard,passed
5,firm:FinCode,passed
6,firm:CodeFinQA,passed
7,firm:CodeTAT-QA,passed
8,firm:SEC-NUM,passed
9,firm:TAT-QA,passed


Verified at UTC: 2026-10-17T07:12:49.478651+00:00
Gateway compatibility usage: {'attempts': 4, 'completed': 4, 'input_tokens': 111, 'output_tokens': 65, 'missing_usage_responses': 0}


,installed version
openai,2.54.0
openai-agents,0.20.0
mcp,1.30.0
bptk-py,2.4.1
sentence-transformers,6.1.0
faiss-cpu,1.15.1
openpyxl,3.1.5
numpy,2.3.1
pandas,2.3.0


## Local credentials

Only presence is shown. Credentials are never printed or embedded in notebook outputs. A fresh public clone has no API credentials.

In [4]:
from dotenv import dotenv_values
local = dotenv_values(ROOT / '.env')
print('Local API key configured:', bool(local.get('SHOPAIKEY_API_KEY')))
ignored = subprocess.run(['git', 'check-ignore', '.env'], cwd=ROOT, capture_output=True, text=True)
print('.env ignored by Git:', ignored.returncode == 0)
local.clear()

Local API key configured: True
.env ignored by Git: True


## Run commands

Use the terminal from the project root. `check` is local and blocks LLM calls. `check_gateway.py` only lists model access; adding `--smoke` makes four short paid calls. A `run` command launches one real paid task. Review its cost and logs before starting long Retail horizons. The 6,000-call/4,096-output-token ceilings are not a monetary spending cap.

In [5]:
python = str(ROOT / '.venv/Scripts/python.exe') if sys.platform == 'win32' else str(ROOT / '.venv/bin/python')
for command in [
    'scripts/run_benchmarks.py check',
    'scripts/check_gateway.py',
    'scripts/run_benchmarks.py run --task firm:CodeFinQA',
]:
    print(('& ' if sys.platform == 'win32' else '') + f'"{python}" {command}')
# Paid runs are deliberately not executed by Run All in this notebook.

& "A:\paper\llm-business-shocks-sim\.venv\Scripts\python.exe" scripts/run_benchmarks.py check
& "A:\paper\llm-business-shocks-sim\.venv\Scripts\python.exe" scripts/check_gateway.py
& "A:\paper\llm-business-shocks-sim\.venv\Scripts\python.exe" scripts/run_benchmarks.py run --task firm:CodeFinQA


## Compatibility notes to retain in the report

- Vendor source versions are pinned and source files remain unchanged.
- Windows FormulaEval replaces fork/signals with a separate process and timeout; the native scorer is retained.
- Factset is a local binary-decision extension and removes provenance label hints. Report it separately from the ten native reasoning datasets.
- EDT retains the native agents/equations and scenario decision, then runs the full ABM through BPTK; it does not make 96 LLM decisions.
- ICBE stops on API/parse failures rather than inventing cooperative actions.
- Current scientific results remain the Kaggle rule-based baselines; the native scripted checks above validate setup only.